In [1]:
# Imports

from pathlib import Path
import matplotlib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Config

DATA_PATH = Path("../data/vgchartz-2024.csv")
FIGURES_DIR = Path("figures")
RANDOM_STATE = 42
TEST_SIZE = 0.20
RARE_CATEGORY_MIN_COUNT = 20

In [2]:
# Step 1: Load and inspect the raw dataset
def missing_summary(df):
    
    summary = pd.DataFrame(
        {
            "missing_count": df.isna().sum(),
            "missing_pct": (df.isna().mean() * 100).round(2),
        }
    )
    
    return summary.sort_values("missing_count", ascending=False)


df = pd.read_csv(DATA_PATH) 
FIGURES_DIR.mkdir(exist_ok=True)

print(f"Dataset shape: {df.shape}")
print("\nColumn types:")
print(df.dtypes)
print("\nMissing-value summary:")
print(missing_summary(df))

Dataset shape: (64016, 14)

Column types:
img              object
title            object
console          object
genre            object
publisher        object
developer        object
critic_score    float64
total_sales     float64
na_sales        float64
jp_sales        float64
pal_sales       float64
other_sales     float64
release_date     object
last_update      object
dtype: object

Missing-value summary:
              missing_count  missing_pct
critic_score          57338        89.57
jp_sales              57290        89.49
na_sales              51379        80.26
pal_sales             51192        79.97
other_sales           48888        76.37
last_update           46137        72.07
total_sales           45094        70.44
release_date           7051        11.01
developer                17         0.03
img                       0         0.00
title                     0         0.00
console                   0         0.00
genre                     0         0.00
publisher 

In [3]:
# Step 2: Preprocessing design

# Decision 1: Use regression rows only

def prepare_regression_subset(df):
 
    regression_df = df[df["total_sales"].notna()].copy()

    leakage_columns = ["na_sales", "jp_sales", "pal_sales", "other_sales"]
    id_like_columns = ["img", "title", "last_update"]
    regression_df = regression_df.drop(columns=leakage_columns + id_like_columns)

    return regression_df

regression_df = prepare_regression_subset(df)

print(f"Regression subset shape: {regression_df.shape}")
print("\nMissing-value summary after target filtering:")
print(missing_summary(regression_df))

Regression subset shape: (18922, 7)

Missing-value summary after target filtering:
              missing_count  missing_pct
critic_score          14796        78.19
release_date             90         0.48
developer                 4         0.02
console                   0         0.00
genre                     0         0.00
publisher                 0         0.00
total_sales               0         0.00


In [4]:
# Decision 2: Engineer date features
# convert `release_date` into: `release_year`, `release_month`, and `holiday_release`


def engineer_date_features(df):

    engineered = df.copy()

    engineered["release_date"] = pd.to_datetime(
        engineered["release_date"], errors="coerce"
    )

    engineered["release_year"] = engineered["release_date"].dt.year
    engineered["release_month"] = engineered["release_date"].dt.month
    engineered["holiday_release"] = engineered["release_month"].isin([10, 11, 12])
    engineered["holiday_release"] = engineered["holiday_release"].astype("float")
    engineered = engineered.drop(columns=["release_date"])

    return engineered

regression_df = engineer_date_features(regression_df)

print(regression_df[["release_year", "release_month", "holiday_release"]].head())

   release_year  release_month  holiday_release
0        2013.0            9.0              0.0
1        2014.0           11.0              1.0
2        2002.0           10.0              1.0
3        2013.0            9.0              0.0
4        2015.0           11.0              1.0


In [5]:
# seperate features and target

X = regression_df.drop(columns=["total_sales"])
y = regression_df["total_sales"].copy()

print("Feature columns:")
print(X.columns.tolist())

Feature columns:
['console', 'genre', 'publisher', 'developer', 'critic_score', 'release_year', 'release_month', 'holiday_release']


In [6]:
# train/test split (80/20 with TEST_SIZE=0.2)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")

Training rows: 15137
Test rows: 3785


In [7]:
# Decision 3: Handle missing values and high-cardinality categories
# `developer` has a few missing values, so we fill them as `Unknown`
# `critic_score` is important but heavily missing, so we keep it and let the numeric imputer fill missing values with the training median
# `publisher` and `developer` have high cardinality, so we group rare values into `Other` using the training split only

def collapse_rare_labels(train, test, min_count):
  
    train_filled = train.fillna("Unknown").astype(str)
    test_filled = test.fillna("Unknown").astype(str)

    frequent_values = train_filled.value_counts()
    keep_values = set(frequent_values[frequent_values >= min_count].index)

    train_grouped = train_filled.where(train_filled.isin(keep_values), "Other")
    test_grouped = test_filled.where(test_filled.isin(keep_values), "Other")
    return train_grouped, test_grouped


for column in ["publisher", "developer"]:
    X_train[column], X_test[column] = collapse_rare_labels(
        X_train[column], X_test[column], min_count=RARE_CATEGORY_MIN_COUNT
    )

X_train["critic_score_missing"] = X_train["critic_score"].isna().astype(float)
X_test["critic_score_missing"] = X_test["critic_score"].isna().astype(float)

print("Unique categories after rare-label grouping:")
for column in ["console", "genre", "publisher", "developer"]:
    print(
        f"{column}: train={X_train[column].nunique(dropna=False)}, "
        f"test={X_test[column].nunique(dropna=False)}"
    )

Unique categories after rare-label grouping:
console: train=39, test=31
genre: train=20, test=17
publisher: train=119, test=118
developer: train=153, test=152


In [8]:
# Investigate outliers in the target

q1 = y_train.quantile(0.25)
q3 = y_train.quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr
outlier_rate = (y_train > upper_bound).mean() * 100

print(f"IQR upper bound for training target: {upper_bound:.3f}")
print(f"Share of training rows above the bound: {outlier_rate:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(y_train, bins=50, color="steelblue", edgecolor="black")
axes[0].set_title("Raw Total Sales")
axes[0].set_xlabel("Sales (millions)")
axes[0].set_ylabel("Frequency")

axes[1].hist(np.log1p(y_train), bins=50, color="darkorange", edgecolor="black")
axes[1].set_title("log1p(Total Sales)")
axes[1].set_xlabel("log1p(sales)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "step2_target_distribution.png", dpi=150)
plt.close(fig)

IQR upper bound for training target: 0.830
Share of training rows above the bound: 10.21%


In [16]:
# Build a reusable preprocessing pipeline
# scale numeric features
# using one shared preprocessing object keeps the baseline and advanced models comparable

def build_preprocessor(
    numeric_features, categorical_features):

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "encoder",
                OneHotEncoder(handle_unknown="ignore", sparse_output = False),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_features),
            ("cat", categorical_pipeline, categorical_features),
        ]
    )

numeric_features = [
    "critic_score",
    "critic_score_missing",
    "release_year",
    "release_month",
    "holiday_release",
]

categorical_features = ["console", "genre", "publisher", "developer"]

preprocessor = build_preprocessor(numeric_features, categorical_features)


In [17]:
# Step 3: Baseline 
# Linear Regression as baseline

baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression()),
    ]
)

baseline_model.fit(X_train, y_train_log)
baseline_predictions_log = baseline_model.predict(X_test)

baseline_predictions = np.expm1(baseline_predictions_log)

baseline_results = {
    "RMSE": mean_squared_error(y_test, baseline_predictions) ** 0.5,
    "MAE": mean_absolute_error(y_test, baseline_predictions),
    "R2": r2_score(y_test, baseline_predictions),
    "Min Prediction": baseline_predictions.min(),
    "Max Prediction": baseline_predictions.max(),
}

baseline_results_df = pd.DataFrame([baseline_results]).round(4)
print("Baseline Linear Regression results:")
print(baseline_results_df.to_string(index=False))

Baseline Linear Regression results:
  RMSE    MAE     R2  Min Prediction  Max Prediction
0.7417 0.2697 0.2477         -0.3167           3.521


In [18]:
# Ridge Regression

from sklearn.linear_model import Ridge

ridge_pipeline = Pipeline(steps = [("preprocessor", preprocessor), ("regressor", Ridge())])

ridge_param_grid = {"regressor__alpha": [0.1, 1.0, 10.0, 100.0]}

ridge_search = GridSearchCV(ridge_pipeline, ridge_param_grid, scoring = "neg_root_mean_squared_error", cv = 3, n_jobs = -1, refit = True)

ridge_search.fit(X_train, y_train_log)

ridge_predictions_log = ridge_search.best_estimator_.predict(X_test)
ridge_predictions = np.expm1(ridge_predictions_log)

In [19]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_test, baseline_predictions, alpha=0.35, color="slateblue")
axes[0].plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    linestyle="--",
    color="black",
)
axes[0].set_title("Actual vs Predicted Sales")
axes[0].set_xlabel("Actual total_sales")
axes[0].set_ylabel("Predicted total_sales")

residuals = y_test - baseline_predictions
axes[1].hist(residuals, bins=50, color="seagreen", edgecolor="black")
axes[1].set_title("Baseline Residual Distribution")
axes[1].set_xlabel("Residual (actual - predicted)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "step3_baseline_diagnostics.png", dpi=150)
plt.close(fig)

In [20]:
# Model 1: Support Vector Regression (SVR)

svr_sample_size = min(3000, len(X_train))
y_bins = pd.qcut(y_train, q = 5, duplicates = "drop")

X_train_svr, _, y_train_svr, _ = train_test_split(
    X_train, 
    y_train_log, 
    train_size = svr_sample_size, 
    stratify = y_bins, 
    random_state = RANDOM_STATE
    )

svr_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", SVR(kernel="rbf")),
    ]
)

svr_param_grid = {
    "regressor__C": [0.1, 1.0, 10.0, 100.0],
    "regressor__epsilon": [0.05, 0.1, 0.2],
    "regressor__gamma": ["scale", "auto"]
}

svr_search = GridSearchCV(
    estimator=svr_pipeline,
    param_grid=svr_param_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1,
    refit=True,
)

svr_search.fit(X_train_svr, y_train_svr)

svr_predictions_log = svr_search.best_estimator_.predict(X_test)
svr_predictions = np.expm1(svr_predictions_log)

svr_cv_results = pd.DataFrame(svr_search.cv_results_)[
    [
        "param_regressor__C",
        "param_regressor__epsilon",
        "mean_test_score",
        "std_test_score",
        "rank_test_score",
    ]
].copy()
svr_cv_results["mean_cv_rmse"] = -svr_cv_results["mean_test_score"]
svr_cv_results["std_cv_rmse"] = svr_cv_results["std_test_score"]
svr_cv_results = svr_cv_results.drop(columns=["mean_test_score", "std_test_score"])
svr_cv_results = svr_cv_results.sort_values("rank_test_score")

print("SVR cross-validation results:")
print(svr_cv_results.round(4).to_string(index=False))
print("\nBest SVR parameters:")
print(svr_search.best_params_)

print("SVR raw prediction sample (log):", svr_search.best_estimator_.predict(X_test[:5]))
print("SVR converted:", np.expm1(svr_search.best_estimator_.predict(X_test[:5])))
print("Actual:", y_test[:5].values)

SVR cross-validation results:
 param_regressor__C  param_regressor__epsilon  rank_test_score  mean_cv_rmse  std_cv_rmse
                1.0                      0.05                1        0.2595       0.0048
                1.0                      0.10                2        0.2599       0.0050
              100.0                      0.10                3        0.2605       0.0050
              100.0                      0.05                4        0.2616       0.0050
              100.0                      0.20                5        0.2657       0.0069
               10.0                      0.10                6        0.2666       0.0063
               10.0                      0.20                7        0.2687       0.0060
               10.0                      0.05                8        0.2697       0.0062
                1.0                      0.20                9        0.2699       0.0051
               10.0                      0.10               10        

In [ ]:
# Model 2: Neural Network (`MLPRegressor`)

mlp_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            MLPRegressor(
                max_iter=300,
                early_stopping=True,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

mlp_param_grid = {
    "regressor__hidden_layer_sizes": [(64, 32), (128, 64)],
    "regressor__alpha": [0.0001, 0.001],
    "regressor__learning_rate_init": [0.001, 0.01],
    "regressor__activation": ["relu", "tanh"]
}

mlp_search = GridSearchCV(
    estimator=mlp_pipeline,
    param_grid=mlp_param_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1,
    refit=True,
)

mlp_search.fit(X_train, y_train_log)

mlp_predictions_log = mlp_search.best_estimator_.predict(X_test)
mlp_predictions = np.expm1(mlp_predictions_log)

mlp_cv_results = pd.DataFrame(mlp_search.cv_results_)[
    [
        "param_regressor__hidden_layer_sizes",
        "param_regressor__alpha",
        "mean_test_score",
        "std_test_score",
        "rank_test_score",
    ]
].copy()
mlp_cv_results["mean_cv_rmse"] = -mlp_cv_results["mean_test_score"]
mlp_cv_results["std_cv_rmse"] = mlp_cv_results["std_test_score"]
mlp_cv_results = mlp_cv_results.drop(columns=["mean_test_score", "std_test_score"])
mlp_cv_results = mlp_cv_results.sort_values("rank_test_score")

print("\nMLP cross-validation results:")
print(mlp_cv_results.round(4).to_string(index=False))
print("\nBest MLP parameters:")
print(mlp_search.best_params_)


MLP cross-validation results:
param_regressor__hidden_layer_sizes  param_regressor__alpha  rank_test_score  mean_cv_rmse  std_cv_rmse
                           (64, 32)                  0.0010                1        0.2289       0.0028
                           (64, 32)                  0.0001                2        0.2291       0.0028
                          (128, 64)                  0.0010                3        0.2300       0.0011
                          (128, 64)                  0.0010                4        0.2301       0.0010
                          (128, 64)                  0.0001                5        0.2301       0.0009
                          (128, 64)                  0.0010                6        0.2302       0.0045
                          (128, 64)                  0.0001                7        0.2306       0.0004
                           (64, 32)                  0.0001                8        0.2306       0.0024
                          (128, 6

In [ ]:
# Histogram Gradient Boosting (tuned)
from sklearn.ensemble import HistGradientBoostingRegressor

hgb_pipeline = Pipeline(
    steps=[("preprocessor", preprocessor), ("regressor", HistGradientBoostingRegressor(random_state=RANDOM_STATE))]
)

hgb_param_grid = {
    "regressor__max_iter":         [200, 300],
    "regressor__learning_rate":    [0.05, 0.1],
    "regressor__max_depth":        [3, 5],
    "regressor__min_samples_leaf": [20, 50],
    "regressor__l2_regularization":[0.1, 1.0],
}

hgb_search = GridSearchCV(
    estimator=hgb_pipeline,
    param_grid=hgb_param_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1,
    refit=True,
)

hgb_search.fit(X_train, y_train_log)

hgb_predictions_log = hgb_search.best_estimator_.predict(X_test)
hgb_predictions = np.expm1(hgb_predictions_log)

hgb_cv_results = pd.DataFrame(hgb_search.cv_results_)[
    [
        "param_regressor__max_iter",
        "param_regressor__learning_rate",
        "param_regressor__max_depth",
        "mean_test_score",
        "std_test_score",
        "rank_test_score",
    ]
].copy()
hgb_cv_results["mean_cv_rmse"] = -hgb_cv_results["mean_test_score"]
hgb_cv_results["std_cv_rmse"] = hgb_cv_results["std_test_score"]
hgb_cv_results = hgb_cv_results.drop(columns=["mean_test_score", "std_test_score"])
hgb_cv_results = hgb_cv_results.sort_values("rank_test_score")

print("HGB cross-validation results (top 10):")
print(hgb_cv_results.head(10).round(4).to_string(index=False))
print("\nBest HGB parameters:")
print(hgb_search.best_params_)

In [23]:
# Step 5: Evaluation
# use RMSE, MAE, R^2

def regression_metrics(y_true, y_pred):
    return {
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }

model_predictions = {
    "Linear Regression": baseline_predictions,
    "Ridge Regression": ridge_predictions,
    "SVR": svr_predictions,
    "MLPRegressor": mlp_predictions,
    "HistGradientBoosting": hgb_predictions
}

evaluation_rows = []
for model_name, predictions in model_predictions.items():
    row = {"Model": model_name}
    row.update(regression_metrics(y_test, predictions))
    evaluation_rows.append(row)

evaluation_df = pd.DataFrame(evaluation_rows).sort_values("RMSE").reset_index(drop=True)
print("Held-out test set results:")
print(evaluation_df.round(4).to_string(index=False))

Held-out test set results:
               Model    RMSE    MAE         R2
HistGradientBoosting  0.6653 0.2380     0.3948
                 SVR  0.7235 0.2512     0.2843
   Linear Regression  0.7417 0.2697     0.2477
    Ridge Regression  0.7422 0.2694     0.2468
        MLPRegressor 64.7723 2.6167 -5735.6812


In [ ]:
# Model comparison table
# Lower RMSE and MAE are better. Higher R^2 is better.

evaluation_table = evaluation_df.copy()
evaluation_table[["RMSE", "MAE", "R2"]] = evaluation_table[["RMSE", "MAE", "R2"]].round(4)
print(evaluation_table.to_string(index=False))


# Visual comparison

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

metric_names = ["RMSE", "MAE", "R2"]
colors = ["steelblue", "darkorange", "seagreen"]

for ax, metric_name, color in zip(axes, metric_names, colors):
    ax.bar(evaluation_df["Model"], evaluation_df[metric_name], color=color)
    ax.set_title(metric_name)
    ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "step5_model_comparison_metrics.png", dpi=150)
plt.close(fig)


fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (model_name, predictions) in zip(axes, model_predictions.items()):
    ax.scatter(y_test, predictions, alpha=0.3)
    ax.plot(
        [y_test.min(), y_test.max()],
        [y_test.min(), y_test.max()],
        linestyle="--",
        color="black",
    )
    ax.set_title(model_name)
    ax.set_xlabel("Actual total_sales")
    ax.set_ylabel("Predicted total_sales")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "step5_actual_vs_predicted.png", dpi=150)
plt.close(fig)

               Model    RMSE    MAE         R2
HistGradientBoosting  0.6653 0.2380     0.3948
                 SVR  0.7235 0.2512     0.2843
   Linear Regression  0.7417 0.2697     0.2477
    Ridge Regression  0.7422 0.2694     0.2468
        MLPRegressor 64.7723 2.6167 -5735.6812


In [ ]:
# Overfitting vs underfitting
# compare training and test performance.

analysis_rows = [
    {
        "Model": "Linear Regression",
        "Train_RMSE": mean_squared_error(y_train, np.expm1(baseline_model.predict(X_train))) ** 0.5,
        "Train_MAE": mean_absolute_error(y_train, np.expm1(baseline_model.predict(X_train))),
        "Train_R2": r2_score(y_train, np.expm1(baseline_model.predict(X_train))),
        "Test_RMSE": mean_squared_error(y_test, baseline_predictions) ** 0.5,
        "Test_MAE": mean_absolute_error(y_test, baseline_predictions),
        "Test_R2": r2_score(y_test, baseline_predictions),
    },
    {
        "Model": "Ridge Regression",
        "Train_RMSE": mean_squared_error(y_train, np.expm1(ridge_search.best_estimator_.predict(X_train))) ** 0.5,
        "Train_MAE": mean_absolute_error(y_train, np.expm1(ridge_search.best_estimator_.predict(X_train))),
        "Train_R2": r2_score(y_train, np.expm1(ridge_search.best_estimator_.predict(X_train))),
        "Test_RMSE": mean_squared_error(y_test, ridge_predictions) ** 0.5,
        "Test_MAE": mean_absolute_error(y_test, ridge_predictions),
        "Test_R2": r2_score(y_test, ridge_predictions),
    },
    {
        "Model": "HistGradientBoosting",
        "Train_RMSE": mean_squared_error(y_train, np.expm1(hgb_search.best_estimator_.predict(X_train))) ** 0.5,
        "Train_MAE": mean_absolute_error(y_train, np.expm1(hgb_search.best_estimator_.predict(X_train))),
        "Train_R2": r2_score(y_train, np.expm1(hgb_search.best_estimator_.predict(X_train))),
        "Test_RMSE": mean_squared_error(y_test, hgb_predictions) ** 0.5,
        "Test_MAE": mean_absolute_error(y_test, hgb_predictions),
        "Test_R2": r2_score(y_test, hgb_predictions),
    },    
    {
        "Model": "SVR",
        "Train_RMSE": mean_squared_error(
            y_train_svr, np.expm1(svr_search.best_estimator_.predict(X_train_svr))
        )
        ** 0.5,
        "Train_MAE": mean_absolute_error(
            y_train_svr, np.expm1(svr_search.best_estimator_.predict(X_train_svr))
        ),
        "Train_R2": r2_score(y_train_svr, np.expm1(svr_search.best_estimator_.predict(X_train_svr))),
        "Test_RMSE": mean_squared_error(
            y_test, np.expm1(svr_search.best_estimator_.predict(X_test))
        )
        ** 0.5,
        "Test_MAE": mean_absolute_error(
            y_test, np.expm1(svr_search.best_estimator_.predict(X_test))
        ),
        "Test_R2": r2_score(y_test, np.expm1(svr_search.best_estimator_.predict(X_test))),
    },
    {
        "Model": "MLPRegressor",
        "Train_RMSE": mean_squared_error(
            y_train, np.expm1(mlp_search.best_estimator_.predict(X_train))
        )
        ** 0.5,
        "Train_MAE": mean_absolute_error(
            y_train, np.expm1(mlp_search.best_estimator_.predict(X_train))
        ),
        "Train_R2": r2_score(y_train, np.expm1(mlp_search.best_estimator_.predict(X_train))),
        "Test_RMSE": mean_squared_error(
            y_test, np.expm1(mlp_search.best_estimator_.predict(X_test))
        )
        ** 0.5,
        "Test_MAE": mean_absolute_error(
            y_test, np.expm1(mlp_search.best_estimator_.predict(X_test))
        ),
        "Test_R2": r2_score(y_test, np.expm1(mlp_search.best_estimator_.predict(X_test))),
    },
]

train_test_df = pd.DataFrame(analysis_rows).round(4)
print("Train vs test performance:")
print(train_test_df.to_string(index=False))

Train vs test performance:
            Model  Train_RMSE  Train_MAE  Train_R2  Test_RMSE  Test_MAE  Test_R2
Linear Regression      0.6644     0.2649    0.3017     0.7417    0.2697   0.2477
              SVR      0.2335     0.1033    0.4909     0.7235    0.2512   0.2843
     MLPRegressor      0.5168     0.2134    0.5775     0.6579    0.2421   0.4082


In [26]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(train_test_df["Model"], train_test_df["Train_RMSE"], label="Train")
axes[0].bar(
    train_test_df["Model"],
    train_test_df["Test_RMSE"],
    alpha=0.7,
    label="Test",
)
axes[0].set_title("Train vs Test RMSE")
axes[0].tick_params(axis="x", rotation=20)
axes[0].legend()

axes[1].bar(train_test_df["Model"], train_test_df["Train_R2"], label="Train")
axes[1].bar(
    train_test_df["Model"],
    train_test_df["Test_R2"],
    alpha=0.7,
    label="Test",
)
axes[1].set_title("Train vs Test R^2")
axes[1].tick_params(axis="x", rotation=20)
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "step6_train_test_gap.png", dpi=150)
plt.close(fig)


In [27]:
# Generalization and error analysis for the best model

best_model_predictions = np.expm1(mlp_search.best_estimator_.predict(X_test))

error_analysis_df = X_test.copy()
error_analysis_df["actual_sales"] = y_test.values
error_analysis_df["predicted_sales"] = best_model_predictions
error_analysis_df["residual"] = (
    error_analysis_df["actual_sales"] - error_analysis_df["predicted_sales"]
)
error_analysis_df["abs_error"] = error_analysis_df["residual"].abs()
error_analysis_df["sales_bin"] = pd.qcut(
    error_analysis_df["actual_sales"], q=4, duplicates="drop"
)

quartile_error_df = (
    error_analysis_df.groupby("sales_bin", observed=False)
    .agg(
        count=("actual_sales", "size"),
        mean_actual=("actual_sales", "mean"),
        mean_pred=("predicted_sales", "mean"),
        mae=("abs_error", "mean"),
    )
    .reset_index()
)

print("\nBest-model error by sales quartile:")
print(quartile_error_df.round(4).to_string(index=False))


Best-model error by sales quartile:
     sales_bin  count  mean_actual  mean_pred    mae
(-0.001, 0.03]    965       0.0142     0.0751 0.0830
  (0.03, 0.12]    997       0.0729     0.1667 0.1199
  (0.12, 0.33]    882       0.2115     0.2802 0.1499
 (0.33, 20.32]    941       1.1102     0.6562 0.6211


In [28]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(
    quartile_error_df["sales_bin"].astype(str),
    quartile_error_df["mae"],
    color="mediumpurple",
)
axes[0].set_title("MLP MAE by Sales Quartile")
axes[0].set_xlabel("Actual sales quartile")
axes[0].set_ylabel("Mean absolute error")
axes[0].tick_params(axis="x", rotation=25)

axes[1].scatter(
    error_analysis_df["actual_sales"],
    error_analysis_df["residual"],
    alpha=0.3,
    color="teal",
)
axes[1].axhline(0, linestyle="--", color="black")
axes[1].set_title("MLP Residuals vs Actual Sales")
axes[1].set_xlabel("Actual total_sales")
axes[1].set_ylabel("Residual (actual - predicted)")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "step6_mlp_error_analysis.png", dpi=150)
plt.close(fig)

In [29]:
# looking at largest errors

largest_underpredictions = error_analysis_df.sort_values(
    "residual", ascending=False
)[
    [
        "console",
        "genre",
        "publisher",
        "developer",
        "critic_score",
        "release_year",
        "actual_sales",
        "predicted_sales",
        "residual",
    ]
].head(10)

largest_overpredictions = error_analysis_df.sort_values(
    "residual", ascending=True
)[
    [
        "console",
        "genre",
        "publisher",
        "developer",
        "critic_score",
        "release_year",
        "actual_sales",
        "predicted_sales",
        "residual",
    ]
].head(10)

print("\nLargest underpredictions:")
print(largest_underpredictions.round(4).to_string(index=False))

print("\nLargest overpredictions:")
print(largest_overpredictions.round(4).to_string(index=False))


Largest underpredictions:
console      genre       publisher         developer  critic_score  release_year  actual_sales  predicted_sales  residual
    PS3     Action  Rockstar Games             Other           9.4        2013.0         20.32           4.0555   16.2645
   X360     Action  Rockstar Games             Other           NaN        2013.0         15.86           0.6075   15.2525
    PS4     Sports Electronic Arts         EA Canada           8.9        2016.0         10.94           3.1589    7.7811
     PC Simulation Electronic Arts EA Redwood Shores           8.5        2009.0          7.96           0.3510    7.6090
   X360    Shooter      Activision          Treyarch           8.8        2010.0         14.74           7.6109    7.1291
    PSP     Action  Rockstar Games             Other           8.8        2005.0          7.72           1.9483    5.7717
   XOne    Shooter      Activision             Other           NaN        2017.0          6.23           0.7565    5.47